# Engineering Toolkit: pytest, CLI Tools, FastAPI, Packaging and ML Tracking

Notebook for **Chapter 23** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch23.html

Tip: change a number, predict the output, then run the cell again.

## 23.1 The Engineering Toolkit: What, When, Install, Import and the Mental Model

In [ ]:
import importlib.util
import importlib.metadata as md
import sys

print("python", sys.version.split()[0])
libs = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "joblib": "joblib",
        "yaml": "pyyaml", "httpx": "httpx", "pytest": "pytest", "fastapi": "fastapi",
        "typer": "typer", "mlflow": "mlflow"}
for module, dist in libs.items():
    if importlib.util.find_spec(module) is None:
        print(f"{module:8s} not installed")
    else:
        print(f"{module:8s} {md.version(dist)}")

python 3.14.3
numpy    2.5.3
pandas   3.0.6
sklearn  1.9.1
joblib   1.6.0
yaml     6.0.3
httpx    0.28.1
pytest   not installed
fastapi  not installed
typer    not installed
mlflow   not installed


## 23.2 Testing: pytest, unittest, Fixtures, Parametrize, Markers and Mocking

A test is a small function that calls your code with a known input and checks the answer. When you change code next month, the tests tell you in seconds whether you broke something. **pytest** is the standard runner: tests are plain functions and plain `assert` statements. It is **not installed** on the study machine, so for every pytest example below you see the pytest version labelled *not run here*, followed by the same checks written with **unittest** (standard library), which really ran and printed the output shown. The ideas (arrange, act, assert) are identical.

In [ ]:
# needs a full environment: pip install prices
# test_prices.py   -- NOT RUN HERE: pytest is not installed
from prices import clean_price

def test_plain_number():
    assert clean_price("12") == 12.0

def test_dollar_and_comma():
    assert clean_price("$1,299.50") == 1299.5

def test_blank_is_none():
    assert clean_price("  ") is None

In [ ]:
import io
import unittest

def clean_price(text):
    text = text.strip().replace("$", "").replace(",", "")
    return float(text) if text else None

class TestPrices(unittest.TestCase):
    def test_plain_number(self):
        self.assertEqual(clean_price("12"), 12.0)

    def test_dollar_and_comma(self):
        self.assertEqual(clean_price("$1,299.50"), 1299.5)

    def test_blank_is_none(self):
        self.assertIsNone(clean_price("  "))

def run(case):
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(case)
    res = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)
    print("ran", res.testsRun, "failures", len(res.failures), "errors", len(res.errors))
    return res

run(TestPrices)

ran 3 failures 0 errors 0


In [ ]:
# needs a full environment: pip install prices pytest
# NOT RUN HERE: pytest is not installed
import pytest
from prices import clean_price

@pytest.mark.parametrize(
    "text, expected",
    [
        ("12", 12.0),
        ("$1,299.50", 1299.5),
        ("  7.5 ", 7.5),
        pytest.param("$0", 1.0, id="wrong-on-purpose"),
    ],
)
def test_clean_price(text, expected):
    assert clean_price(text) == expected

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import io
import unittest

def clean_price(text):
    text = text.strip().replace("$", "").replace(",", "")
    return float(text) if text else None

class TestPrices(unittest.TestCase):
    def test_table(self):
        cases = [("12", 12.0), ("$1,299.50", 1299.5), ("  7.5 ", 7.5), ("$0", 1.0)]
        for text, expected in cases:
            with self.subTest(text=text):
                self.assertEqual(clean_price(text), expected)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestPrices)
res = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)
print("failures:", len(res.failures))
case, message = res.failures[0]
print(str(case))
print(message.strip().splitlines()[-1])

failures: 1
test_table (__main__.TestPrices.test_table) (text='$0')
AssertionError: 0.0 != 1.0


In [ ]:
# needs a full environment: pip install pandas pytest
# NOT RUN HERE: pytest is not installed
import pandas as pd
import pytest

@pytest.fixture
def csv_file(tmp_path):
    path = tmp_path / "orders.csv"
    path.write_text("order_id,amount\n1,10.5\n2,4.5\n")
    yield path            # the test runs here
    print("teardown")     # code after yield runs afterwards

def test_total(csv_file):
    df = pd.read_csv(csv_file)
    assert df["amount"].sum() == 15.0

In [ ]:
import io
import tempfile
import unittest
from pathlib import Path

import pandas as pd

events = []

class TestOrders(unittest.TestCase):
    def setUp(self):
        events.append("setup")
        tmp = tempfile.TemporaryDirectory()
        self.addCleanup(lambda: events.append("teardown"))
        self.addCleanup(tmp.cleanup)
        self.path = Path(tmp.name) / "orders.csv"
        self.path.write_text("order_id,amount\n1,10.5\n2,4.5\n")

    def test_total(self):
        events.append("test_total")
        df = pd.read_csv(self.path)
        self.assertEqual(df["amount"].sum(), 15.0)

    def test_rows(self):
        events.append("test_rows")
        self.assertEqual(len(pd.read_csv(self.path)), 2)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestOrders)
res = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print(res.testsRun, "tests, ok:", res.wasSuccessful())
print(events)

2 tests, ok: True
['setup', 'test_rows', 'teardown', 'setup', 'test_total', 'teardown']


In [ ]:
# needs a full environment: pip install pytest unittest
# NOT RUN HERE: pytest is not installed
import os
from unittest.mock import Mock

def get_key():
    return os.environ["API_KEY"]

def score_all(client, ids):
    return [client.predict(i) for i in ids]

def test_get_key(monkeypatch):
    monkeypatch.setenv("API_KEY", "test-key")      # undone after the test
    assert get_key() == "test-key"

def test_missing_key(monkeypatch):
    monkeypatch.delenv("API_KEY", raising=False)
    import pytest
    with pytest.raises(KeyError):
        get_key()

def test_score_all():
    client = Mock()
    client.predict.side_effect = [0.1, 0.9]
    assert score_all(client, [7, 8]) == [0.1, 0.9]
    client.predict.assert_any_call(8)

In [ ]:
import os
import unittest
from unittest import mock
from unittest.mock import Mock

def get_key():
    return os.environ["API_KEY"]

def score_all(client, ids):
    return [client.predict(i) for i in ids]

# 1. replace an environment variable for the length of a with-block
with mock.patch.dict(os.environ, {"API_KEY": "test-key"}):
    print(get_key())
print("API_KEY" in os.environ)

# 2. a Mock records calls and returns scripted values
client = Mock()
client.predict.side_effect = [0.1, 0.9]
print(score_all(client, [7, 8]))
print(client.predict.call_count, client.predict.call_args_list)
client.predict.assert_any_call(8)

# 3. patch an attribute of a real module
import math
with mock.patch.object(math, "sqrt", return_value=42.0):
    print(math.sqrt(4))
print(math.sqrt(4))

test-key
False
[0.1, 0.9]
2 [call(7), call(8)]
42.0
2.0


In [ ]:
import io
import unittest

import pandas as pd
from pandas.testing import assert_frame_equal

def add_revenue(df):
    if "price" not in df.columns or "qty" not in df.columns:
        raise KeyError("need price and qty")
    out = df.copy()
    out["revenue"] = out["price"] * out["qty"]
    return out

class TestRevenue(unittest.TestCase):
    def test_values(self):
        df = pd.DataFrame({"price": [2.5, 4.0], "qty": [2, 3]})
        want = pd.DataFrame({"price": [2.5, 4.0], "qty": [2, 3], "revenue": [5.0, 12.0]})
        assert_frame_equal(add_revenue(df), want)

    def test_input_not_modified(self):
        df = pd.DataFrame({"price": [1.0], "qty": [1]})
        add_revenue(df)
        self.assertEqual(list(df.columns), ["price", "qty"])

    def test_missing_column(self):
        with self.assertRaises(KeyError):
            add_revenue(pd.DataFrame({"price": [1.0]}))

    def test_wrong_dtype_is_caught(self):
        df = pd.DataFrame({"price": [2.5], "qty": [2]})
        bad = pd.DataFrame({"price": [2.5], "qty": [2], "revenue": [5]})   # int, not float
        with self.assertRaises(AssertionError) as cm:
            assert_frame_equal(add_revenue(df), bad)
        print(str(cm.exception).splitlines()[0])

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestRevenue)
res = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print(res.testsRun, "tests, ok:", res.wasSuccessful())

Attributes of DataFrame.iloc[:, 2] (column name="revenue") are different
4 tests, ok: True


## 23.3 Logging, argparse, Typer/Click and pathlib: Jobs That Run Without You

A job that runs at 3 a.m. has no screen. You need three things: a record of what happened (logging), a way to give it inputs (a command-line interface) and paths that do not break when the folder changes (pathlib).

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, stream=sys.stdout,
                    format="%(levelname)s %(name)s: %(message)s")
log = logging.getLogger("etl")

rows = 1250
log.debug("this is hidden, level is INFO")
log.info("loaded %d rows", rows)
log.warning("column %s has %.1f%% missing", "age", 12.34)
log.error("could not write %s", "out.csv")

INFO etl: loaded 1250 rows
WARNING etl: column age has 12.3% missing
ERROR etl: could not write out.csv


In [ ]:
import io
import logging

console, logfile = io.StringIO(), io.StringIO()
log = logging.getLogger("train")
log.setLevel(logging.DEBUG)
log.propagate = False

h1 = logging.StreamHandler(console)
h1.setLevel(logging.WARNING)
h1.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
h2 = logging.StreamHandler(logfile)          # stands in for FileHandler
h2.setLevel(logging.DEBUG)
h2.setFormatter(logging.Formatter("%(name)s %(levelname)-7s %(message)s"))
log.addHandler(h1)
log.addHandler(h2)

log.debug("fold 1 started")
log.info("fold 1 accuracy=0.81")
log.warning("fold 2 has few positives")

print("CONSOLE:")
print(console.getvalue(), end="")
print("FILE:")
print(logfile.getvalue(), end="")

CONSOLE:
WARNING | fold 2 has few positives
FILE:
train DEBUG   fold 1 started
train INFO    fold 1 accuracy=0.81
train WARNING fold 2 has few positives


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import logging
import logging.config
import sys

class ShortExc(logging.Formatter):
    """Keep only the last traceback line (full paths differ on every machine)."""
    def formatException(self, exc_info):
        return super().formatException(exc_info).splitlines()[-1]

config = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {"plain": {"()": ShortExc, "format": "%(levelname)s [%(name)s] %(message)s"}},
    "handlers": {"out": {"class": "logging.StreamHandler", "stream": "ext://sys.stdout",
                         "formatter": "plain", "level": "INFO"}},
    "loggers": {"pipeline": {"level": "DEBUG", "handlers": ["out"], "propagate": False}},
}
logging.config.dictConfig(config)
log = logging.getLogger("pipeline.clean")     # a child of "pipeline"

def parse_age(text):
    try:
        return int(text)
    except ValueError:
        log.exception("bad age %r", text)
        return None

print(parse_age("41"))
print(parse_age("forty"))

41
ERROR [pipeline.clean] bad age 'forty'
ValueError: invalid literal for int() with base 10: 'forty'
None


In [ ]:
import argparse

p = argparse.ArgumentParser(prog="clean", description="Clean a CSV file")
p.add_argument("csv", help="input file")
p.add_argument("--out", default="clean.csv", help="output file")
p.add_argument("--min-rows", type=int, default=1, help="fail if fewer rows")
p.add_argument("--dry-run", action="store_true")
p.add_argument("-v", "--verbose", action="count", default=0)

ns = p.parse_args(["orders.csv", "--min-rows", "100", "-vv"])
print(ns)
print(ns.min_rows + 1, ns.dry_run, ns.verbose)

Namespace(csv='orders.csv', out='clean.csv', min_rows=100, dry_run=False, verbose=2)
101 False 2


In [ ]:
import argparse
import contextlib
import io

def make_parser():
    p = argparse.ArgumentParser(prog="mlcli")
    sub = p.add_subparsers(dest="cmd", required=True)
    t = sub.add_parser("train", help="train a model")
    t.add_argument("--C", type=float, default=1.0)
    t.add_argument("--model", choices=["logreg", "tree"], default="logreg")
    t.set_defaults(func=lambda a: f"training {a.model} with C={a.C}")
    e = sub.add_parser("eval", help="evaluate a model")
    e.add_argument("run_id")
    e.set_defaults(func=lambda a: f"evaluating run {a.run_id}")
    return p

p = make_parser()
for argv in (["train", "--C", "0.5"], ["eval", "r42"]):
    a = p.parse_args(argv)
    print(a.func(a))

err = io.StringIO()
with contextlib.redirect_stderr(err):
    try:
        p.parse_args(["train", "--model", "svm"])
    except SystemExit as exc:
        print("exit status", exc.code)
print(err.getvalue().strip().splitlines()[-1])

training logreg with C=0.5
evaluating run r42
exit status 2
mlcli train: error: argument --model: invalid choice: 'svm' (choose from logreg, tree)


In [ ]:
# needs a full environment: pip install typer
# NOT RUN HERE: typer is not installed
from pathlib import Path
from typing import Annotated

import typer

app = typer.Typer(help="Tools for order files")

@app.command()
def clean(
    csv: Annotated[Path, typer.Argument(exists=True, help="input CSV")],
    out: Annotated[Path, typer.Option(help="output file")] = Path("clean.csv"),
    min_rows: Annotated[int, typer.Option(min=1)] = 1,
    dry_run: bool = False,
):
    """Remove duplicate rows."""
    typer.echo(f"cleaning {csv} -> {out} (dry_run={dry_run})")

@app.command()
def version():
    typer.echo("1.0")

if __name__ == "__main__":
    app()

In [ ]:
# needs a full environment: pip install click
# NOT RUN HERE: click is not installed
import click

@click.command()
@click.argument("csv", type=click.Path(exists=True))
@click.option("--out", default="clean.csv", show_default=True)
@click.option("--min-rows", type=int, default=1)
@click.option("--dry-run", is_flag=True)
def clean(csv, out, min_rows, dry_run):
    """Remove duplicate rows."""
    click.echo(f"cleaning {csv} -> {out}")

if __name__ == "__main__":
    clean()

In [ ]:
from pathlib import Path, PurePosixPath

p = PurePosixPath("/data/raw/orders_2026.csv.gz")
print(p.name)
print(p.stem)
print(p.suffix, p.suffixes)
print(p.parent)
print(p.with_suffix(""))
print(p.parents[1])

root = PurePosixPath("/data")
print(p.relative_to(root))
print(root / "clean" / "orders.parquet")

orders_2026.csv.gz
orders_2026.csv
.gz ['.csv', '.gz']
/data/raw
/data/raw/orders_2026.csv
/data
raw/orders_2026.csv.gz
/data/clean/orders.parquet


In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "raw" / "2026").mkdir(parents=True)
    (root / "raw" / "a.csv").write_text("x\n1\n")
    (root / "raw" / "2026" / "b.csv").write_text("x\n1\n2\n")
    (root / "raw" / "notes.txt").write_text("hello")

    print(sorted(p.name for p in (root / "raw").iterdir()))
    print(sorted(str(p.relative_to(root)) for p in root.rglob("*.csv")))
    print([p.stat().st_size for p in sorted(root.rglob("*.csv"))])

    # safe write: write to a temporary name, then atomically replace
    target = root / "report.json"
    tmp_file = target.with_suffix(".json.tmp")
    tmp_file.write_text('{"ok": true}')
    tmp_file.replace(target)
    print(target.read_text(), tmp_file.exists())

['2026', 'a.csv', 'notes.txt']
['raw/2026/b.csv', 'raw/a.csv']
[6, 4]
{"ok": true} False


## 23.4 dataclasses, typing, mypy, Environments, pre-commit and Secrets

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from dataclasses import dataclass, field, asdict, replace

@dataclass(frozen=True, kw_only=True)
class TrainConfig:
    csv: str
    lr: float = 0.1
    epochs: int = 20
    seed: int = 0
    features: tuple[str, ...] = field(default_factory=tuple)

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f"lr must be positive, got {self.lr}")

cfg = TrainConfig(csv="train.csv", features=("age", "tenure"))
print(cfg)
print(asdict(cfg)["epochs"])
cfg2 = replace(cfg, lr=0.01)
print(cfg2.lr, cfg.lr, cfg == cfg2)

try:
    cfg.lr = 5
except Exception as exc:
    print(type(exc).__name__)
try:
    TrainConfig(csv="x.csv", lr=-1)
except ValueError as exc:
    print(exc)
try:
    TrainConfig(csv="x.csv", learning_rate=0.5)
except TypeError as exc:
    print("TypeError")

TrainConfig(csv='train.csv', lr=0.1, epochs=20, seed=0, features=('age', 'tenure'))
20
0.01 0.1 False
FrozenInstanceError
lr must be positive, got -1
TypeError


In [ ]:
import typing

def mean(values: list[float]) -> float:
    return sum(values) / len(values)

print(mean([1.0, 2.0, 6.0]))
print(mean((1, 2, 6)))            # a tuple of ints: works, hints are not enforced
print(typing.get_type_hints(mean))

def find(name: str, table: dict[str, int]) -> int | None:
    return table.get(name)

result = find("b", {"a": 1})
print(result)
if result is not None:
    print(result + 1)

3.0
3.0
{'values': list[float], 'return': <class 'float'>}
None


In [ ]:
import tomllib

text = """
[project]
name = "churn-tools"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = ["pandas>=2.2", "scikit-learn>=1.5", "typer>=0.12"]

[project.optional-dependencies]
dev = ["pytest>=8", "mypy"]

[project.scripts]
churn = "churn_tools.cli:app"

[tool.pytest.ini_options]
testpaths = ["tests"]
"""
data = tomllib.loads(text)
proj = data["project"]
print(proj["name"], proj["version"])
print(proj["dependencies"])
print(proj["optional-dependencies"]["dev"])
print(proj["scripts"])
print(data["tool"]["pytest"]["ini_options"]["testpaths"])

churn-tools 0.1.0
['pandas>=2.2', 'scikit-learn>=1.5', 'typer>=0.12']
['pytest>=8', 'mypy']
{'churn': 'churn_tools.cli:app'}
['tests']


In [ ]:
import subprocess
import sys
import tempfile
import venv
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    env_dir = Path(tmp) / ".venv"
    venv.EnvBuilder(with_pip=False).create(env_dir)       # with_pip=False keeps it offline and quick
    print((env_dir / "pyvenv.cfg").exists())
    py = env_dir / "bin" / "python"                         # Windows: Scripts\\python.exe
    code = "import sys; print(sys.prefix != sys.base_prefix)"
    out = subprocess.run([str(py), "-c", code], capture_output=True, text=True)
    print(out.stdout.strip())
    out = subprocess.run([str(py), "-c", "import pandas"], capture_output=True, text=True)
    print(out.returncode, out.stderr.strip().splitlines()[-1])

True
True
1 ModuleNotFoundError: No module named 'pandas'


In [ ]:
import os
from dataclasses import dataclass, field
from unittest import mock

@dataclass(frozen=True)
class Settings:
    db_url: str
    api_token: str = field(repr=False)          # never shown in repr, logs or tracebacks

    @classmethod
    def from_env(cls):
        missing = [k for k in ("DB_URL", "API_TOKEN") if not os.environ.get(k)]
        if missing:
            raise RuntimeError("missing environment variables: " + ", ".join(missing))
        return cls(os.environ["DB_URL"], os.environ["API_TOKEN"])

def mask(secret, keep=2):
    return secret[:keep] + "*" * (len(secret) - keep)

with mock.patch.dict(os.environ, {"DB_URL": "sqlite:///demo.db", "API_TOKEN": "s3cr3t-token"}, clear=True):
    s = Settings.from_env()
print(s)
print(mask(s.api_token))

with mock.patch.dict(os.environ, {"DB_URL": "sqlite:///demo.db"}, clear=True):
    try:
        Settings.from_env()
    except RuntimeError as exc:
        print(exc)

Settings(db_url='sqlite:///demo.db')
s3**********
missing environment variables: API_TOKEN


## 23.5 Serving and Speed: FastAPI, a Standard-Library API, and joblib

In [ ]:
# needs a full environment: pip install contextlib fastapi joblib pydantic
# app.py  -- NOT RUN HERE: fastapi, pydantic and uvicorn are not installed
from contextlib import asynccontextmanager
from pathlib import Path

import joblib
from fastapi import FastAPI
from pydantic import BaseModel, Field

MODEL_PATH = Path("model.joblib")
state = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    state["model"] = joblib.load(MODEL_PATH)      # load once, not per request
    yield
    state.clear()

app = FastAPI(title="Churn API", lifespan=lifespan)

class Customer(BaseModel):
    age: int = Field(ge=18, le=100)
    tenure_months: float = Field(ge=0)
    monthly_spend: float = Field(gt=0)

class Prediction(BaseModel):
    churn_probability: float
    label: int

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=Prediction)
def predict(c: Customer):
    x = [[c.age, c.tenure_months, c.monthly_spend]]
    p = float(state["model"].predict_proba(x)[0, 1])
    return Prediction(churn_probability=round(p, 4), label=int(p >= 0.5))

In [ ]:
# needs a full environment: pip install app fastapi
# test_app.py  -- NOT RUN HERE
from fastapi.testclient import TestClient
from app import app

def test_health():
    with TestClient(app) as client:               # the with-block runs lifespan
        r = client.get("/health")
        assert r.status_code == 200
        assert r.json() == {"status": "ok"}

def test_predict_ok():
    with TestClient(app) as client:
        r = client.post("/predict", json={"age": 40, "tenure_months": 12, "monthly_spend": 30.5})
        assert r.status_code == 200
        body = r.json()
        assert 0.0 <= body["churn_probability"] <= 1.0
        assert body["label"] in (0, 1)

def test_predict_rejects_young_customer():
    with TestClient(app) as client:
        r = client.post("/predict", json={"age": 17, "tenure_months": 1, "monthly_spend": 5})
        assert r.status_code == 422
        assert r.json()["detail"][0]["loc"][-1] == "age"

def test_predict_missing_field():
    with TestClient(app) as client:
        r = client.post("/predict", json={"age": 30})
        assert r.status_code == 422

In [ ]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

import httpx
import numpy as np
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)
X = np.column_stack([rng.integers(18, 80, 400), rng.uniform(0, 60, 400), rng.uniform(5, 100, 400)])
y = ((X[:, 1] < 12) & (X[:, 2] > 40)).astype(int)
model = LogisticRegression(max_iter=2000).fit(X / [100, 60, 100], y)

RULES = {"age": (18, 100), "tenure_months": (0, 1000), "monthly_spend": (0.01, 10_000)}

def validate(body):
    problems = []
    for name, (lo, hi) in RULES.items():
        v = body.get(name)
        if not isinstance(v, (int, float)) or isinstance(v, bool):
            problems.append({"loc": name, "msg": "field required or not a number"})
        elif not lo <= v <= hi:
            problems.append({"loc": name, "msg": f"must be between {lo} and {hi}"})
    return problems

class Handler(BaseHTTPRequestHandler):
    def _send(self, code, payload):
        data = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        if self.path == "/health":
            self._send(200, {"status": "ok"})
        else:
            self._send(404, {"detail": "not found"})

    def do_POST(self):
        if self.path != "/predict":
            return self._send(404, {"detail": "not found"})
        raw = self.rfile.read(int(self.headers.get("Content-Length", 0)))
        try:
            body = json.loads(raw)
        except json.JSONDecodeError:
            return self._send(400, {"detail": "invalid JSON"})
        problems = validate(body)
        if problems:
            return self._send(422, {"detail": problems})
        x = np.array([[body["age"], body["tenure_months"], body["monthly_spend"]]]) / [100, 60, 100]
        p = float(model.predict_proba(x)[0, 1])
        self._send(200, {"churn_probability": round(p, 4), "label": int(p >= 0.5)})

    def log_message(self, *args):          # keep the console quiet
        pass

server = HTTPServer(("127.0.0.1", 0), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
base = f"http://127.0.0.1:{server.server_port}"

with httpx.Client(base_url=base, timeout=30, trust_env=False) as client:
    r = client.get("/health")
    print(r.status_code, r.json())
    r = client.post("/predict", json={"age": 40, "tenure_months": 3, "monthly_spend": 80})
    print(r.status_code, r.json())
    r = client.post("/predict", json={"age": 40, "tenure_months": 50, "monthly_spend": 10})
    print(r.status_code, r.json())
    r = client.post("/predict", json={"age": 17, "tenure_months": 1, "monthly_spend": 5})
    print(r.status_code, r.json())
    r = client.post("/predict", content=b"not json")
    print(r.status_code, r.json())
    r = client.get("/nothing")
    print(r.status_code)
server.shutdown()

200 {'status': 'ok'}
200 {'churn_probability': 0.6531, 'label': 1}
200 {'churn_probability': 0.0034, 'label': 0}
422 {'detail': [{'loc': 'age', 'msg': 'must be between 18 and 100'}]}
400 {'detail': 'invalid JSON'}
404


In [ ]:
import tempfile
from pathlib import Path

import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(1)
X = rng.normal(size=(300, 5))
y = (X[:, 0] + X[:, 1] > 0).astype(int)
model = RandomForestClassifier(n_estimators=50, random_state=0).fit(X, y)

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "model.joblib"
    joblib.dump(model, path)
    small = Path(tmp) / "model_c.joblib"
    joblib.dump(model, small, compress=3)
    print(path.stat().st_size > small.stat().st_size)
    loaded = joblib.load(path)

print(np.array_equal(model.predict(X), loaded.predict(X)))
print(type(loaded).__name__, loaded.n_estimators)

True
True
RandomForestClassifier 50


In [ ]:
import tempfile

from joblib import Memory

calls = []

def slow_square(x):
    calls.append(x)               # a side effect, so we can count real executions
    return x * x

with tempfile.TemporaryDirectory() as tmp:
    memory = Memory(tmp, verbose=0)
    cached = memory.cache(slow_square)
    print(cached(12), cached(12), cached(13))
    print("real calls:", calls)
    cached.clear(warn=False)
    cached(12)
    print("after clear:", calls)

144 144 169
real calls: [12, 13]
after clear: [12, 13, 12]


In [ ]:
import time
from joblib import Parallel, delayed

def tiny(i):
    return i * i

def heavy(n):
    s = 0
    for k in range(n):
        s += k % 7
    return s

def clock(f):
    t = time.perf_counter()
    f()
    return time.perf_counter() - t

print(Parallel(n_jobs=2)(delayed(tiny)(i) for i in range(6)))

clock(lambda: Parallel(n_jobs=4)(delayed(tiny)(i) for i in range(4)))      # warm up the workers
for name, fn, arg in (("tiny", tiny, 5), ("heavy", heavy, 2_000_000)):
    serial = clock(lambda: [fn(arg) for _ in range(16)])
    par = clock(lambda: Parallel(n_jobs=4)(delayed(fn)(arg) for _ in range(16)))
    print(name, "parallel faster:", par < serial)

[0, 1, 4, 9, 16, 25]
tiny parallel faster: False
heavy parallel faster: True


## 23.6 Tracking, Versioning, Shipping and Scheduling: MLflow, DVC, Docker, Airflow and Prefect

This section is about **awareness**: what each tool does, the five lines you need to start, and a runnable stand-in built from the standard library so you understand the idea even without installing the tool. MLflow, DVC, Docker, Airflow and Prefect are all **not run here**.

In [ ]:
import json
import tempfile
import time
import uuid
from contextlib import contextmanager
from pathlib import Path

import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

@contextmanager
def start_run(folder, name):
    record = {"run_id": uuid.uuid4().hex[:8], "name": name, "params": {}, "metrics": {}}
    started = time.time()
    try:
        yield record
        record["status"] = "FINISHED"
    except Exception:
        record["status"] = "FAILED"
        raise
    finally:
        record["seconds"] = round(time.time() - started, 3)
        (Path(folder) / f"{record['run_id']}.json").write_text(json.dumps(record))

X, y = make_classification(n_samples=600, n_features=10, n_informative=4, flip_y=0.1, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

with tempfile.TemporaryDirectory() as tmp:
    for C in (0.001, 0.01, 0.1, 1, 10):
        with start_run(tmp, f"logreg-C{C}") as run:
            run["params"]["C"] = C
            m = LogisticRegression(C=C, max_iter=1000).fit(Xtr, ytr)
            run["metrics"]["train_acc"] = round(m.score(Xtr, ytr), 4)
            run["metrics"]["test_acc"] = round(m.score(Xte, yte), 4)
    runs = [json.loads(p.read_text()) for p in Path(tmp).glob("*.json")]

df = pd.json_normalize(runs).sort_values("params.C")
print(df[["name", "params.C", "metrics.train_acc", "metrics.test_acc", "status"]].to_string(index=False))
best = df.loc[df["metrics.test_acc"].idxmax()]
print("best:", best["name"])

         name  params.C  metrics.train_acc  metrics.test_acc   status
logreg-C0.001     0.001             0.6310            0.6333 FINISHED
 logreg-C0.01     0.010             0.7048            0.6778 FINISHED
  logreg-C0.1     0.100             0.7024            0.6667 FINISHED
    logreg-C1     1.000             0.7048            0.6611 FINISHED
   logreg-C10    10.000             0.7048            0.6667 FINISHED
best: logreg-C0.01


In [ ]:
# needs a full environment: pip install mlflow sklearn
# NOT RUN HERE: mlflow is not installed. Check the docs for your version
# (older releases used log_model(model, artifact_path="model")).
import mlflow
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=600, n_features=10, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

mlflow.set_experiment("churn")
for C in (0.01, 0.1, 1):
    with mlflow.start_run(run_name=f"logreg-C{C}"):
        mlflow.log_params({"model": "logreg", "C": C, "seed": 0})
        model = LogisticRegression(C=C, max_iter=1000).fit(Xtr, ytr)
        mlflow.log_metric("test_acc", model.score(Xte, yte))
        mlflow.sklearn.log_model(model, name="model")

In [ ]:
# needs a full environment: pip install airflow
# NOT RUN HERE: apache-airflow is not installed. Airflow 3 style; in Airflow 2 import from airflow.decorators.
from datetime import datetime
from airflow.sdk import dag, task

@dag(schedule="0 6 * * 1", start_date=datetime(2026, 1, 1), catchup=False, tags=["churn"])
def weekly_churn():
    @task(retries=2)
    def extract() -> str:
        return "data/raw.csv"

    @task
    def validate(path: str) -> str:
        return path

    @task
    def train(path: str) -> float:
        return 0.81

    @task
    def report(score: float):
        print(f"score={score}")

    report(train(validate(extract())))

weekly_churn()

In [ ]:
# needs a full environment: pip install prefect
# NOT RUN HERE: prefect is not installed (Prefect 3 style)
from prefect import flow, task

@task(retries=3, retry_delay_seconds=60)
def extract() -> str:
    return "data/raw.csv"

@task
def train(path: str) -> float:
    return 0.81

@flow(log_prints=True)
def weekly_churn():
    path = extract()
    score = train(path)
    print(f"score={score}")

if __name__ == "__main__":
    weekly_churn()                                  # run once now
    # weekly_churn.serve(name="weekly", cron="0 6 * * 1")   # or run on a schedule

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from graphlib import TopologicalSorter

deps = {                       # task: the tasks it depends on
    "extract": set(),
    "validate": {"extract"},
    "features": {"validate"},
    "baseline": {"validate"},
    "evaluate": {"features", "baseline"},
    "report": {"evaluate"},
}
attempts = {"baseline": 0}

def run_task(name):
    if name == "baseline":
        attempts["baseline"] += 1
        if attempts["baseline"] < 3:
            raise RuntimeError("flaky source")
    return f"{name} ok"

ts = TopologicalSorter(deps)
ts.prepare()
wave = 0
while ts.is_active():
    ready = list(ts.get_ready())
    print(f"wave {wave}: {sorted(ready)}")
    for name in sorted(ready):
        for attempt in range(1, 4):
            try:
                run_task(name)
                break
            except RuntimeError:
                print(f"  {name} failed on attempt {attempt}")
        ts.done(name)
    wave += 1

try:
    TopologicalSorter({"a": {"b"}, "b": {"a"}}).prepare()
except Exception as exc:
    print(type(exc).__name__)

wave 0: ['extract']
wave 1: ['validate']
wave 2: ['baseline', 'features']
  baseline failed on attempt 1
  baseline failed on attempt 2
wave 3: ['evaluate']
wave 4: ['report']
CycleError


## 23.7 Four Complete Recipes

Now the pieces work together. Each recipe is a complete, runnable solution to a task you will really meet.

In [ ]:
import argparse
import io
import sys
import tempfile
import unittest
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path

import pandas as pd

def build_parser():
    p = argparse.ArgumentParser(prog="csvtool", description="Sum a value column by group")
    p.add_argument("csv", type=Path)
    p.add_argument("--group", required=True)
    p.add_argument("--value", required=True)
    p.add_argument("--top", type=int, default=3)
    return p

def top_groups(df, group, value, top):
    for col in (group, value):
        if col not in df.columns:
            raise KeyError(f"column {col!r} not found; have {list(df.columns)}")
    return df.groupby(group)[value].sum().sort_values(ascending=False).head(top)

def main(argv=None):
    args = build_parser().parse_args(argv)
    if not args.csv.is_file():
        print(f"error: file not found: {args.csv}", file=sys.stderr)
        return 2
    try:
        result = top_groups(pd.read_csv(args.csv), args.group, args.value, args.top)
    except KeyError as exc:
        print(f"error: {exc.args[0]}", file=sys.stderr)
        return 2
    print(result.to_string())
    return 0

class TestCsvTool(unittest.TestCase):
    def setUp(self):
        tmp = tempfile.TemporaryDirectory()
        self.addCleanup(tmp.cleanup)
        self.csv = Path(tmp.name) / "sales.csv"
        self.csv.write_text("region,sales\nnorth,10\nsouth,30\nnorth,25\neast,5\nsouth,1\n")

    def run_main(self, *argv):
        out, err = io.StringIO(), io.StringIO()
        with redirect_stdout(out), redirect_stderr(err):
            code = main([str(a) for a in argv])
        return code, out.getvalue(), err.getvalue()

    def test_happy_path(self):
        code, out, _ = self.run_main(self.csv, "--group", "region", "--value", "sales", "--top", 2)
        self.assertEqual(code, 0)
        self.assertEqual(out.split(), ["region", "north", "35", "south", "31"])

    def test_unknown_column(self):
        code, _, err = self.run_main(self.csv, "--group", "city", "--value", "sales")
        self.assertEqual(code, 2)
        self.assertIn("column 'city' not found", err)

    def test_missing_file(self):
        code, _, err = self.run_main("nope.csv", "--group", "a", "--value", "b")
        self.assertEqual(code, 2)
        self.assertIn("file not found", err)

    def test_pure_function(self):
        df = pd.DataFrame({"g": ["a", "b", "a"], "v": [1, 5, 2]})
        self.assertEqual(top_groups(df, "g", "v", 1).to_dict(), {"b": 5})

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCsvTool)
res = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print(res.testsRun, "tests, ok:", res.wasSuccessful())

with tempfile.TemporaryDirectory() as tmp:
    f = Path(tmp) / "sales.csv"
    f.write_text("region,sales\nnorth,10\nsouth,30\nnorth,25\neast,5\nsouth,1\n")
    print("exit code:", main([str(f), "--group", "region", "--value", "sales", "--top", "2"]))

4 tests, ok: True
region
north    35
south    31
exit code: 0


In [ ]:
# needs a full environment: pip install csvtool
# test_csvtool.py -- NOT RUN HERE: the same tests in pytest style
from csvtool import main

def test_happy_path(tmp_path, capsys):
    f = tmp_path / "sales.csv"
    f.write_text("region,sales\nnorth,10\nsouth,30\nnorth,25\n")
    assert main([str(f), "--group", "region", "--value", "sales", "--top", "1"]) == 0
    assert "north" in capsys.readouterr().out

def test_unknown_column(tmp_path, capsys):
    f = tmp_path / "s.csv"
    f.write_text("a,b\n1,2\n")
    assert main([str(f), "--group", "zzz", "--value", "b"]) == 2
    assert "not found" in capsys.readouterr().err

In [ ]:
import json
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression

FEATURES = ["age", "tenure_months", "monthly_spend"]

def save_bundle(model, folder, metrics):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, folder / "model.joblib")
    meta = {"features": FEATURES, "sklearn": sklearn.__version__, "metrics": metrics}
    (folder / "meta.json").write_text(json.dumps(meta, indent=2))

def load_bundle(folder):
    folder = Path(folder)
    meta = json.loads((folder / "meta.json").read_text())
    if meta["sklearn"] != sklearn.__version__:
        raise RuntimeError(f"model built with scikit-learn {meta['sklearn']}, running {sklearn.__version__}")
    return joblib.load(folder / "model.joblib"), meta

def predict(model, meta, rows):
    df = pd.DataFrame(rows)
    missing = set(meta["features"]) - set(df.columns)
    if missing:
        raise ValueError(f"missing features: {sorted(missing)}")
    return model.predict_proba(df[meta["features"]].to_numpy())[:, 1]     # enforce order

rng = np.random.default_rng(0)
X = pd.DataFrame({"age": rng.integers(18, 80, 300), "tenure_months": rng.uniform(0, 60, 300),
                  "monthly_spend": rng.uniform(5, 100, 300)})
y = ((X["tenure_months"] < 12) & (X["monthly_spend"] > 40)).astype(int)
model = LogisticRegression(max_iter=2000).fit(X[FEATURES].to_numpy() / [100, 60, 100], y)

with tempfile.TemporaryDirectory() as tmp:
    save_bundle(model, Path(tmp) / "v1", {"train_acc": round(model.score(X[FEATURES].to_numpy() / [100, 60, 100], y), 3)})
    loaded, meta = load_bundle(Path(tmp) / "v1")
    print(sorted(p.name for p in (Path(tmp) / "v1").iterdir()))
    print(meta["features"], meta["metrics"])
    meta_bad = json.loads((Path(tmp) / "v1" / "meta.json").read_text())
    meta_bad["sklearn"] = "0.0.1"
    (Path(tmp) / "v1" / "meta.json").write_text(json.dumps(meta_bad))
    try:
        load_bundle(Path(tmp) / "v1")
    except RuntimeError as exc:
        print("blocked:", str(exc)[:42])

try:
    predict(loaded, meta, [{"age": 30, "tenure_months": 2}])
except ValueError as exc:
    print(exc)

['meta.json', 'model.joblib']
['age', 'tenure_months', 'monthly_spend'] {'train_acc': 0.897}
blocked: model built with scikit-learn 0.0.1, runni
missing features: ['monthly_spend']


In [ ]:
import argparse
import hashlib
import json
import platform
import tempfile
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

@dataclass(frozen=True)
class Config:
    C: float = 1.0
    seed: int = 0
    n_samples: int = 800

def fingerprint(*arrays):
    h = hashlib.sha256()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:12]

def train(cfg):
    X, y = make_classification(n_samples=cfg.n_samples, n_features=12, n_informative=5,
                               flip_y=0.05, random_state=cfg.seed)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=cfg.seed, stratify=y)
    model = LogisticRegression(C=cfg.C, max_iter=1000, random_state=cfg.seed).fit(Xtr, ytr)
    proba = model.predict_proba(Xte)[:, 1]
    return {
        "params": asdict(cfg),
        "data_hash": fingerprint(X, y),
        "metrics": {"accuracy": round(accuracy_score(yte, proba > 0.5), 4),
                    "roc_auc": round(roc_auc_score(yte, proba), 4)},
        "env": {"python": platform.python_version(), "sklearn": sklearn.__version__, "numpy": np.__version__},
    }

def main(argv=None, out_dir="."):
    p = argparse.ArgumentParser()
    p.add_argument("--C", type=float, default=1.0)
    p.add_argument("--seed", type=int, default=0)
    a = p.parse_args(argv)
    record = train(Config(C=a.C, seed=a.seed))
    path = Path(out_dir) / "runs.jsonl"
    with path.open("a") as f:
        f.write(json.dumps(record, sort_keys=True) + "\n")
    return record

with tempfile.TemporaryDirectory() as tmp:
    r1 = main(["--C", "0.1", "--seed", "3"], tmp)
    r2 = main(["--C", "0.1", "--seed", "3"], tmp)
    r3 = main(["--C", "0.1", "--seed", "4"], tmp)
    lines = (Path(tmp) / "runs.jsonl").read_text().splitlines()

print(len(lines), "records appended")
print(r1["metrics"], r1["data_hash"])
print("same seed identical:", r1 == r2)
print("other seed differs:", r1["metrics"] != r3["metrics"], r1["data_hash"] != r3["data_hash"])
print(sorted(json.loads(lines[0])))

3 records appended
{'accuracy': 0.715, 'roc_auc': 0.7759} 4ff7770f41e0
same seed identical: True
other seed differs: True True
['data_hash', 'env', 'metrics', 'params']


In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

FILES = {
    "pyproject.toml": '[project]\nname = "churn-tools"\nversion = "0.1.0"\nrequires-python = ">=3.11"\n'
                      'dependencies = ["pandas"]\n\n[project.scripts]\nchurn = "churn_tools.cli:main"\n',
    "README.md": "# churn-tools\n\nInstall: `uv sync` or `pip install -e .[dev]`. Test: `pytest`.\n",
    ".gitignore": ".venv/\n.env\n__pycache__/\n*.pkl\nmlruns/\ndata/raw/\n",
    "src/churn_tools/__init__.py": '__version__ = "0.1.0"\n',
    "src/churn_tools/cli.py": 'def add(a, b):\n    return a + b\n\ndef main():\n    print(add(2, 3))\n',
    "tests/test_cli.py": "import unittest\nfrom churn_tools.cli import add\n\nclass T(unittest.TestCase):\n"
                         "    def test_add(self):\n        self.assertEqual(add(2, 3), 5)\n",
    "data/.gitkeep": "",
}

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp) / "churn-tools"
    for rel, text in FILES.items():
        path = root / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text)
    for p in sorted(root.rglob("*")):
        depth = len(p.relative_to(root).parts) - 1
        print("  " * depth + p.name + ("/" if p.is_dir() else ""))
    env = {**os.environ, "PYTHONPATH": str(root / "src")}
    r = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"],
                       cwd=root, env=env, capture_output=True, text=True)
    print(r.returncode, r.stderr.strip().splitlines()[-1])

.gitignore
README.md
data/
  .gitkeep
pyproject.toml
src/
  churn_tools/
    __init__.py
    cli.py
tests/
  test_cli.py
0 OK


## 23.8 Recipes: "How Do I ...?"

Short, solved, runnable answers to questions that come up every week. Each is standalone.

In [ ]:
from dataclasses import dataclass, fields
import yaml

@dataclass(frozen=True)
class Config:
    csv: str
    C: float = 1.0
    seed: int = 0

text = "csv: data/train.csv\nC: 0.5\n"
raw = yaml.safe_load(text)
cfg = Config(**raw)
print(cfg)
try:
    Config(**yaml.safe_load("csv: a.csv\nlearning_rate: 0.1\n"))
except TypeError as exc:
    print("unknown key rejected:", "learning_rate" in str(exc))
print([f.name for f in fields(Config)])

Config(csv='data/train.csv', C=0.5, seed=0)
unknown key rejected: True
['csv', 'C', 'seed']


In [ ]:
import functools
import logging
import sys

logging.basicConfig(level=logging.INFO, stream=sys.stdout, format="%(levelname)s %(message)s")
log = logging.getLogger("retry")

def retry(times=3, exceptions=(ConnectionError,)):
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as exc:
                    log.warning("%s failed (attempt %d/%d): %s", fn.__name__, attempt, times, exc)
                    if attempt == times:
                        raise
        return wrapper
    return deco

state = {"n": 0}

@retry(times=3)
def fetch():
    state["n"] += 1
    if state["n"] < 3:
        raise ConnectionError("timeout")
    return "data"

print(fetch())

WARNING fetch failed (attempt 1/3): timeout
WARNING fetch failed (attempt 2/3): timeout
data


In [ ]:
import tempfile
from pathlib import Path

import pandas as pd

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "big.csv"
    pd.DataFrame({"user": range(10), "amount": [5.0] * 10}).to_csv(path, index=False)
    total, rows = 0.0, 0
    for i, chunk in enumerate(pd.read_csv(path, chunksize=4)):
        total += chunk["amount"].sum()
        rows += len(chunk)
        print(f"chunk {i}: {len(chunk)} rows")
print(rows, total)

chunk 0: 4 rows
chunk 1: 4 rows
chunk 2: 2 rows
10 50.0


In [ ]:
import hashlib
import pandas as pd

def df_hash(df):
    row_hashes = pd.util.hash_pandas_object(df, index=True).to_numpy()
    return hashlib.sha256(row_hashes.tobytes()).hexdigest()[:10]

a = pd.DataFrame({"x": [1, 2, 3], "y": ["a", "b", "c"]})
b = a.copy()
c = a.copy()
c.loc[1, "x"] = 99
print(df_hash(a) == df_hash(b), df_hash(a) == df_hash(c))
print(len(df_hash(a)))

True False
10


In [ ]:
import httpx

def get_user_name(client, user_id):
    r = client.get(f"/users/{user_id}")
    r.raise_for_status()
    return r.json()["name"]

def handler(request):
    if request.url.path == "/users/1":
        return httpx.Response(200, json={"name": "Ada"})
    return httpx.Response(404, json={"detail": "not found"})

client = httpx.Client(transport=httpx.MockTransport(handler), base_url="http://api.test")
print(get_user_name(client, 1))
try:
    get_user_name(client, 2)
except httpx.HTTPStatusError as exc:
    print(exc.response.status_code)

Ada
404


In [ ]:
import os
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    for i, name in enumerate(["export_01.csv", "export_02.csv", "notes.txt", "export_03.csv"]):
        f = root / name
        f.write_text("x")
        os.utime(f, (1_700_000_000 + i * 100, 1_700_000_000 + i * 100))   # set modified times
    newest = max(root.glob("export_*.csv"), key=lambda p: p.stat().st_mtime)
    print(newest.name)
    print(sorted(p.name for p in root.glob("export_*.csv"))[-1])

export_03.csv
export_03.csv


In [ ]:
import tempfile

import numpy as np
import pandas as pd
from joblib import Memory

counter = {"n": 0}

def build_features(n_rows, seed):
    counter["n"] += 1
    rng = np.random.default_rng(seed)
    return pd.DataFrame({"a": rng.normal(size=n_rows)}).assign(b=lambda d: d["a"] ** 2)

with tempfile.TemporaryDirectory() as tmp:
    cached = Memory(tmp, verbose=0).cache(build_features)
    f1 = cached(1000, 7)
    f2 = cached(1000, 7)
    f3 = cached(1000, 8)
print(counter["n"], f1.equals(f2), f1.equals(f3), f1.shape)

2 True False (1000, 2)


In [ ]:
import io
import pandas as pd

log = io.StringIO(
    '{"run": "a", "C": 0.1, "acc": 0.81}\n'
    '{"run": "b", "C": 1.0, "acc": 0.84}\n'
    '{"run": "c", "C": 10.0, "acc": 0.83}\n'
)
df = pd.read_json(log, lines=True)
print({c: str(t) for c, t in df.dtypes.items()})
print(df.sort_values("acc", ascending=False).head(1).to_string(index=False))

{'run': 'str', 'C': 'float64', 'acc': 'float64'}
run   C  acc
  b 1.0 0.84


In [ ]:
import json
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

@dataclass
class RunInfo:
    data: Path
    started: datetime
    C: float

def encode(obj):
    if isinstance(obj, Path):
        return obj.as_posix()
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"cannot serialise {type(obj).__name__}")

info = RunInfo(Path("data/train.csv"), datetime(2026, 11, 9, 6, 0), 0.5)
text = json.dumps(asdict(info), default=encode)
print(text)
print(json.loads(text)["started"][:10])

{"data": "data/train.csv", "started": "2026-11-09T06:00:00", "C": 0.5}
2026-11-09


In [ ]:
import pandas as pd

EXPECTED = {"customer_id": "int64", "plan": "str", "monthly_spend": "float64"}

def check_schema(df):
    problems = []
    for col, dtype in EXPECTED.items():
        if col not in df.columns:
            problems.append(f"missing column {col}")
        elif str(df[col].dtype) != dtype:
            problems.append(f"{col}: expected {dtype}, got {df[col].dtype}")
    if "monthly_spend" in df and (df["monthly_spend"] < 0).any():
        problems.append("monthly_spend has negative values")
    return problems

good = pd.DataFrame({"customer_id": [1, 2], "plan": ["a", "b"], "monthly_spend": [9.5, 20.0]})
bad = pd.DataFrame({"customer_id": ["1", "2"], "monthly_spend": [-1.0, 3.0]})
print(check_schema(good))
for p in check_schema(bad):
    print(p)

[]
customer_id: expected int64, got str
missing column plan
monthly_spend has negative values


In [ ]:
import os

def parse_env(text):
    values = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values

sample = '# demo\nDB_URL="sqlite:///demo.db"\nDEBUG=1\n\nTOKEN = abc=def\n'
env = parse_env(sample)
print(env)
for k, v in env.items():
    os.environ.setdefault(k, v)           # real environment variables win
print(os.environ["DEBUG"])

{'DB_URL': 'sqlite:///demo.db', 'DEBUG': '1', 'TOKEN': 'abc=def'}
1


In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "-c", "print('hello from child')"],
                        capture_output=True, text=True, check=True, timeout=30)
print(result.stdout.strip(), result.returncode)

try:
    subprocess.run([sys.executable, "-c", "import sys; sys.exit(3)"], check=True)
except subprocess.CalledProcessError as exc:
    print("failed with", exc.returncode)

hello from child 0
failed with 3


## 23.9 Performance and Pitfalls

In [ ]:
def add_tag_wrong(tag, tags=[]):
    tags.append(tag)
    return tags

def add_tag(tag, tags=None):
    tags = [] if tags is None else tags
    return [*tags, tag]

print(add_tag_wrong("a"), add_tag_wrong("b"))      # symptom: "b" call remembers "a"
print(add_tag("a"), add_tag("b"))

['a', 'b'] ['a', 'b']
['a'] ['b']


In [ ]:
import math
import types
from unittest import mock

m = types.ModuleType("m")                  # a stand-in module that did "from math import sqrt"
m.sqrt = math.sqrt
m.root = lambda x: m.sqrt(x)

with mock.patch("math.sqrt", return_value=-1):        # wrong: patches the original only
    print(m.root(9))
with mock.patch.object(m, "sqrt", return_value=-1):   # right: patch the name the code looks up
    print(m.root(9))

3.0
-1


In [ ]:
import io
import logging

buf = io.StringIO()
logging.basicConfig(stream=buf, level=logging.INFO, format="%(message)s")   # root handler

def get_logger_wrong():
    lg = logging.getLogger("job")
    lg.setLevel(logging.INFO)
    lg.addHandler(logging.StreamHandler(buf))      # a new handler on every call
    return lg

get_logger_wrong().info("start")
get_logger_wrong().info("start")
print(buf.getvalue().count("start"))

buf.truncate(0); buf.seek(0)
lg = logging.getLogger("job2")                     # fix: no handler here, rely on the root handler
lg.setLevel(logging.INFO)
lg.info("start")
lg.info("start")
print(buf.getvalue().count("start"))

5
2


In [ ]:
import os
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp).resolve()
    (root / "data").mkdir()
    (root / "data" / "train.csv").write_text("x\n1\n")
    script = root / "job.py"
    script.write_text("")                    # imagine this is the file containing the code below

    HERE = script.resolve().parent           # fix: anchor to the file, not to the cwd
    os.chdir(tempfile.gettempdir())          # simulate running from somewhere else
    print(Path("data/train.csv").exists())
    print((HERE / "data" / "train.csv").exists())

False
True


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
X = np.column_stack([rng.normal(50, 10, 400), rng.normal(0, 1, 400)])
y = (X[:, 0] > 50).astype(int)

scaler = StandardScaler().fit(X)
lonely = LogisticRegression().fit(scaler.transform(X), y)       # wrong to ship this alone
pipe = make_pipeline(StandardScaler(), LogisticRegression()).fit(X, y)

row = np.array([[45.0, 0.0]])                                   # clearly class 0 (45 is below 50)
print(lonely.predict(row), pipe.predict(row))
print(round(lonely.predict_proba(row)[0, 1], 3), round(pipe.predict_proba(row)[0, 1], 3))

[1] [0]
1.0 0.038


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(1)
train = pd.DataFrame({"age": rng.normal(0, 1, 300), "spend": rng.normal(0, 1, 300)})
y = (train["spend"] > 0).astype(int)                         # only "spend" matters
model = LogisticRegression().fit(train.to_numpy(), y)         # names are lost here

request = {"spend": 2.0, "age": -1.0}                         # dictionary in a different order
wrong = model.predict(pd.DataFrame([request]).to_numpy())     # columns: spend, age (swapped)
right = model.predict(pd.DataFrame([request])[["age", "spend"]].to_numpy())
print(wrong, right)

named = LogisticRegression().fit(train, y)                    # fitted on a DataFrame: remembers names
print(list(named.feature_names_in_))

[0] [1]
['age', 'spend']


In [ ]:
import argparse

p = argparse.ArgumentParser()
p.add_argument("--shuffle", type=bool, default=True)        # wrong
print(p.parse_args(["--shuffle", "False"]).shuffle)

q = argparse.ArgumentParser()
q.add_argument("--shuffle", action=argparse.BooleanOptionalAction, default=True)
print(q.parse_args(["--no-shuffle"]).shuffle, q.parse_args([]).shuffle)

True
False True


In [ ]:
import math
import numpy as np

print(0.1 + 0.2 == 0.3, 0.1 + 0.2)
print(math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9))
print(np.allclose([0.1 + 0.2], [0.3]))

False 0.30000000000000004
True
True


In [ ]:
from datetime import datetime, timedelta

def is_stale(updated, now=None):
    now = now or datetime.now()              # real code uses the clock; tests inject a fixed one
    return now - updated > timedelta(days=7)

fixed_now = datetime(2026, 11, 9, 12, 0)
print(is_stale(datetime(2026, 11, 1), now=fixed_now))
print(is_stale(datetime(2026, 11, 5), now=fixed_now))

True
False


In [ ]:
import tempfile
from joblib import Memory

SETTINGS = {"rate": 2}

def scaled(x):                                   # wrong: depends on a global
    return x * SETTINGS["rate"]

def scaled_ok(x, rate):                          # right: every input is an argument
    return x * rate

with tempfile.TemporaryDirectory() as tmp:
    mem = Memory(tmp, verbose=0)
    a, b = mem.cache(scaled), mem.cache(scaled_ok)
    print(a(10))
    SETTINGS["rate"] = 3
    print(a(10))                                 # symptom: still 20
    print(b(10, 2), b(10, 3))

20
20
20 30


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X = np.arange(20).reshape(10, 2)
y = np.arange(10)
a = train_test_split(y, test_size=0.3)[1]
b = train_test_split(y, test_size=0.3)[1]
c = train_test_split(y, test_size=0.3, random_state=42)[1]
d = train_test_split(y, test_size=0.3, random_state=42)[1]
print(np.array_equal(c, d))
print(len(a), len(b))

True
3 3


In [ ]:
# needs a full environment: pip install the libraries used below
# NOT RUN HERE
@app.post("/predict")
async def predict(c: Customer):           # wrong: async, but the body is blocking CPU work
    return model.predict_proba(to_row(c))

@app.post("/predict")
def predict(c: Customer):                  # right: plain def runs in a worker thread
    return model.predict_proba(to_row(c))

In [ ]:
import io
import unittest

SHARED = []

class Bad(unittest.TestCase):
    def test_a_adds(self):
        SHARED.append(1)
        self.assertEqual(len(SHARED), 1)

    def test_b_adds(self):
        SHARED.append(2)
        self.assertEqual(len(SHARED), 1)       # passes only if it runs first

class Good(unittest.TestCase):
    def setUp(self):
        self.items = []                         # fresh state for every test

    def test_a_adds(self):
        self.items.append(1)
        self.assertEqual(len(self.items), 1)

    def test_b_adds(self):
        self.items.append(2)
        self.assertEqual(len(self.items), 1)

for case in (Bad, Good):
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(case)
    res = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
    print(case.__name__, "failures:", len(res.failures))

Bad failures: 1
Good failures: 0


## 23.10 Cross-Library Integration

The tools in this chapter are designed to snap together. The table shows the seams where most real work happens.

In [ ]:
import io
import logging
import tempfile
from dataclasses import dataclass
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

stream = io.StringIO()
logging.basicConfig(stream=stream, level=logging.INFO, format="%(levelname)s %(message)s", force=True)
log = logging.getLogger("score_job")

@dataclass(frozen=True)
class JobConfig:
    model_path: Path
    input_csv: Path
    output_csv: Path
    threshold: float = 0.5

def run(cfg):
    model = joblib.load(cfg.model_path)
    df = pd.read_csv(cfg.input_csv)
    log.info("read %d rows from %s", len(df), cfg.input_csv.name)
    if df.isna().any().any():
        log.error("input has missing values")
        return 1
    df["score"] = model.predict_proba(df[["age", "spend"]].to_numpy())[:, 1].round(3)
    df["flag"] = (df["score"] >= cfg.threshold).astype(int)
    tmp = cfg.output_csv.with_suffix(".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(cfg.output_csv)
    log.info("wrote %d rows, %d flagged", len(df), df["flag"].sum())
    return 0

with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    rng = np.random.default_rng(0)
    Xt = np.column_stack([rng.uniform(18, 80, 300), rng.uniform(5, 100, 300)])
    yt = (Xt[:, 1] > 60).astype(int)
    joblib.dump(make_pipeline(StandardScaler(), LogisticRegression()).fit(Xt, yt), d / "m.joblib")

    pd.DataFrame({"age": [30, 45, 52, 61], "spend": [10.0, 90.0, 70.0, 20.0]}).to_csv(d / "in.csv", index=False)
    cfg = JobConfig(d / "m.joblib", d / "in.csv", d / "out.csv")
    print("exit", run(cfg))
    print(pd.read_csv(d / "out.csv")[["age", "spend", "flag"]].to_string(index=False))
    print(sorted(p.name for p in d.iterdir()))

    pd.DataFrame({"age": [30, None], "spend": [10.0, 90.0]}).to_csv(d / "bad.csv", index=False)
    print("exit", run(JobConfig(d / "m.joblib", d / "bad.csv", d / "out2.csv")))
print(stream.getvalue(), end="")

exit 0
 age  spend  flag
  30   10.0     0
  45   90.0     1
  52   70.0     1
  61   20.0     0
['in.csv', 'm.joblib', 'out.csv']
exit 1
INFO read 4 rows from in.csv
INFO wrote 4 rows, 2 flagged
INFO read 2 rows from bad.csv
ERROR input has missing values


## 23.11 How to Solve Any Engineering and MLOps Problem: A Playbook

In [ ]:
import importlib.metadata as md

REQUIRED = {"numpy": (2, 0), "pandas": (3, 0), "scikit-learn": (1, 5), "not-a-real-package": (1, 0)}

def parse(version):
    return tuple(int(p) for p in version.split(".")[:2])

for dist, minimum in REQUIRED.items():
    try:
        have = md.version(dist)
    except md.PackageNotFoundError:
        print(f"{dist}: MISSING")
        continue
    status = "ok" if parse(have) >= minimum else f"TOO OLD (need {minimum[0]}.{minimum[1]})"
    print(f"{dist}: {have} {status}")

numpy: 2.5.3 ok
pandas: 3.0.6 ok
scikit-learn: 1.9.1 ok
not-a-real-package: MISSING


In [ ]:
import io
import random
import unittest

STATE = {"count": 0}

class T(unittest.TestCase):
    def test_one(self):
        STATE["count"] += 1
        self.assertEqual(STATE["count"], 1)         # hidden dependence on order

    def test_two(self):
        STATE["count"] += 1
        self.assertGreaterEqual(STATE["count"], 1)

tests = [T("test_one"), T("test_two")]
outcomes = []
for seed in range(4):
    STATE["count"] = 0
    order = tests[:]
    random.Random(seed).shuffle(order)
    res = unittest.TextTestRunner(stream=io.StringIO()).run(unittest.TestSuite(order))
    outcomes.append((tuple(t._testMethodName for t in order), len(res.failures)))
for order, failures in outcomes:
    print(order, "failures:", failures)
print("order dependent:", len({f for _, f in outcomes}) > 1)

('test_one', 'test_two') failures: 0
('test_two', 'test_one') failures: 1
('test_two', 'test_one') failures: 1
('test_two', 'test_one') failures: 1
order dependent: True


In [ ]:
import pandas as pd

def check_stage(name, df, min_rows, key=None):
    if len(df) < min_rows:
        raise ValueError(f"{name}: only {len(df)} rows, expected at least {min_rows}")
    if key is not None and df[key].duplicated().any():
        raise ValueError(f"{name}: duplicate values in {key}")
    print(f"{name}: {len(df)} rows ok")

orders = pd.DataFrame({"order_id": [1, 2, 3, 4], "customer": ["a", "b", "z", "a"]})
customers = pd.DataFrame({"customer": ["a", "b"], "segment": ["gold", "basic"]})

check_stage("orders", orders, 4, key="order_id")
joined = orders.merge(customers, on="customer", how="inner")       # silently drops customer "z"
try:
    check_stage("joined", joined, min_rows=len(orders))
except ValueError as exc:
    print("caught:", exc)

orders: 4 rows ok
caught: joined: only 3 rows, expected at least 4


In [ ]:
import cProfile
import pstats

def slow_unique(items):
    out = []
    for x in items:
        if x not in out:                  # list membership is linear: O(n) each time
            out.append(x)
    return out

def fast_unique(items):
    return list(dict.fromkeys(items))     # hash lookup: O(1) each time, keeps order

def pipeline():
    data = [i % 500 for i in range(5000)]
    return slow_unique(data), fast_unique(data)

prof = cProfile.Profile()
prof.runcall(pipeline)
stats = pstats.Stats(prof)
rows = {k[2]: v for k, v in stats.stats.items() if k[2] in ("slow_unique", "fast_unique", "pipeline")}
print({name: rows[name][1] for name in sorted(rows)})
print(slow_unique([3, 1, 3, 2]) == fast_unique([3, 1, 3, 2]))

{'fast_unique': 1, 'pipeline': 1, 'slow_unique': 1}
True


In [ ]:
import io
import unittest

def parse_amount(text):
    text = text.replace("$", "").replace(",", "").strip()
    if text.startswith("(") and text.endswith(")"):        # the fix: accounting negatives "(12.50)"
        return -float(text[1:-1])
    return float(text)

class TestRegression(unittest.TestCase):
    def test_accounting_negative_from_incident_2291(self):
        self.assertEqual(parse_amount("($12.50)"), -12.5)

    def test_old_behaviour_still_works(self):
        self.assertEqual(parse_amount("$1,200.00"), 1200.0)

res = unittest.TextTestRunner(stream=io.StringIO()).run(unittest.defaultTestLoader.loadTestsFromTestCase(TestRegression))
print(res.testsRun, res.wasSuccessful())

2 True


## 23.12 Going Deeper: Internals, Memory, Speed and What Interviewers Probe

This part is for when you want to know why things behave as they do. You can skip it on a first read and return later.

In [ ]:
import inspect

FIXTURES, builds = {}, []

def fixture(fn):
    FIXTURES[fn.__name__] = fn
    return fn

@fixture
def base():
    builds.append("base")
    return 10

@fixture
def doubled(base):
    builds.append("doubled")
    return base * 2

def resolve(name, cache):
    if name not in cache:
        fn = FIXTURES[name]
        cache[name] = fn(*[resolve(p, cache) for p in inspect.signature(fn).parameters])
    return cache[name]

def run_test(test):
    cache = {}                                    # one cache per test = function scope
    test(*[resolve(p, cache) for p in inspect.signature(test).parameters])

def test_values(doubled, base):
    assert doubled == 2 * base

run_test(test_values)
print("passed; built:", builds)

passed; built: ['base', 'doubled']


In [ ]:
import logging

class Loud:
    calls = 0
    def __str__(self):
        Loud.calls += 1
        return "value"

log = logging.getLogger("lazy")
log.setLevel(logging.INFO)
log.addHandler(logging.NullHandler())
log.propagate = False

log.debug("a %s", Loud())            # DEBUG is below INFO: the record is never formatted
print(Loud.calls)
log.debug(f"a {Loud()}")             # the f-string is evaluated BEFORE the call
print(Loud.calls)
log.info("a %s", Loud())             # emitted, so the argument is formatted once
print(Loud.calls)
print(logging.getLogger("a.b").parent.name, logging.getLogger("a.b.c").parent.name)

0
1
1
root a.b


In [ ]:
import tracemalloc
from dataclasses import dataclass

@dataclass
class Plain:
    x: int
    y: int

@dataclass(slots=True)
class Slotted:
    x: int
    y: int

def traced_bytes(cls, n=20000):
    tracemalloc.start()
    objs = [cls(1, 2) for _ in range(n)]
    current, _ = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return current, len(objs)

plain, _ = traced_bytes(Plain)
slotted, _ = traced_bytes(Slotted)
print(slotted < plain)
print(hasattr(Plain(1, 2), "__dict__"), hasattr(Slotted(1, 2), "__dict__"))

True
True False


## Chapter recap

## Practice set

Twenty-four problems, easiest first. Try each before reading the answer. Code in the problems was run to produce the answers.

In [ ]:
import argparse

p = argparse.ArgumentParser()
p.add_argument("--n", type=int, default="5")
ns = p.parse_args([])
print(ns.n + 1, type(ns.n).__name__)

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class P:
    x: int

a, b = P(1), P(1)
print(a == b, a is b, hash(a) == hash(b))

In [ ]:
from joblib import Parallel, delayed

print(Parallel(n_jobs=2)(delayed(pow)(i, 2) for i in range(5)))

In [ ]:
import logging
import sys

logging.warning("starting")
logging.basicConfig(level=logging.DEBUG, stream=sys.stdout, format="%(message)s")
logging.debug("hello")

In [ ]:
from dataclasses import dataclass

try:
    @dataclass
    class Run:
        name: str
        tags: list = []
except ValueError as exc:
    print(exc)

In [ ]:
# needs a full environment: pip install numbers_utils pytest
import pytest
from numbers_utils import is_even

@pytest.mark.parametrize("n, expected", [(0, True), (2, True), (-4, True), (1, False), (-3, False)])
def test_is_even(n, expected):
    assert is_even(n) is expected

In [ ]:
from pathlib import PurePosixPath

print(PurePosixPath("a") / "b" / "../c")
print(PurePosixPath("/data") / "/etc/passwd")
print(PurePosixPath("a/b/c.txt").parent.parent)

In [ ]:
# needs a full environment: pip install the libraries used below
@app.post("/predict")
def predict(c: Customer):
    model = joblib.load("model.joblib")          # inside the handler
    return {"p": float(model.predict_proba([[c.age, c.spend]])[0, 1])}

In [ ]:
from joblib import Parallel, delayed

seen = []

def record(i):
    seen.append(i)
    return i

out = Parallel(n_jobs=2)(delayed(record)(i) for i in range(4))
print(out, len(seen))

In [ ]:
import io
import sys
from contextlib import contextmanager
from pathlib import Path

@contextmanager
def open_input(arg):
    if arg == "-":
        yield sys.stdin
    else:
        with Path(arg).open() as f:
            yield f

sys.stdin = io.StringIO("a,b\n1,2\n")           # a test replaces stdin by a string buffer
with open_input("-") as f:
    print(f.read().splitlines())

In [ ]:
import io
import logging
import time
from contextlib import contextmanager

buf = io.StringIO()
logging.basicConfig(stream=buf, level=logging.INFO, format="%(message)s", force=True)
log = logging.getLogger("timer")

@contextmanager
def timed(name):
    start = time.perf_counter()
    try:
        yield
    finally:
        log.info("%s took %.3f s", name, time.perf_counter() - start)

with timed("sleep"):
    time.sleep(0.05)
try:
    with timed("boom"):
        raise RuntimeError("fail")
except RuntimeError:
    pass
lines = buf.getvalue().splitlines()
print([line.split(" took")[0] for line in lines])
print(float(lines[0].split()[2]) >= 0.05)

In [ ]:
def merge(*layers):
    """Later layers win; None means 'not set' and is ignored."""
    out = {}
    for layer in layers:
        out.update({k: v for k, v in layer.items() if v is not None})
    return out

defaults = {"C": 1.0, "seed": 0, "out": "runs.jsonl"}
file_cfg = {"C": 0.5}
env_cfg = {"seed": 7, "C": None}
cli_cfg = {"out": "tmp.jsonl", "seed": None}
print(merge(defaults, file_cfg, env_cfg, cli_cfg))

In [ ]:
# needs a full environment: pip install numpy sklearn
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

np.random.seed(0)
X, y = load_data("latest.csv")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2)
model = RandomForestClassifier(n_estimators=200).fit(Xtr, ytr)
print(model.score(Xte, yte))

In [ ]:
import json
import tempfile
from pathlib import Path

def save_results(rows, day, folder):
    target = Path(folder) / f"results_{day}.json"          # the name is derived from the day
    tmp = target.with_suffix(".tmp")
    tmp.write_text(json.dumps(rows))
    tmp.replace(target)                                    # overwrite, never append
    return target

with tempfile.TemporaryDirectory() as d:
    save_results([1, 2, 3], "2026-11-09", d)
    save_results([1, 2, 3], "2026-11-09", d)               # a retry
    print(sorted(p.name for p in Path(d).iterdir()))
    print(json.loads((Path(d) / "results_2026-11-09.json").read_text()))

In [ ]:
import pandas as pd

fixture = pd.DataFrame({"a": [1, 2]})
alias = fixture
alias["b"] = 3
print(list(fixture.columns))

subset = fixture[["a"]]
subset.loc[0, "a"] = 99
print(fixture.loc[0, "a"], subset.loc[0, "a"])

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch23.html